# CNN Replica Notebook

Goal: replicate the paper setup as closely as possible while using CSV amplitude vectors and a 1D CNN instead of FFT images and ResNet50.

Key similarities to the paper:
- Three-class supervised classification: Real, Fake High, Fake Low
- Paper-style six augmentation categories
- 90/10 train/test split
- 5-fold cross-validation
- Optimizer comparison: SGD, Adam, RMSProp
- Learning rate 0.001
- 100 epochs
- Mini-batch size 8

Key differences:
- CSV rows of 512 amplitude values instead of RGB FFT images
- Custom 1D CNN instead of pre-trained ResNet50


In [1]:
# ==============================
# Block 1: Data loading / preprocessing / augmentation
# ==============================

import os
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# ------------------------------
# Reproducibility
# ------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ------------------------------
# Paths
# ------------------------------

RAW_DATA_DIR = Path("Kefra_Processed_Data")
AUG_DATA_DIR = Path("Kefra_Augmented_Data")
AUG_SCRIPT = Path("augment_kefra_csv.py")

USE_AUGMENTED_DATA = True
RUN_AUGMENTATION_SCRIPT = True

# This matches the paper's reported 6x augmentation factor.
# If include-original is set in the script, it becomes 7x instead.

if USE_AUGMENTED_DATA and RUN_AUGMENTATION_SCRIPT:
    if not AUG_SCRIPT.exists():
        raise FileNotFoundError(
            "augment_kefra_csv.py not found in the current directory. "
            "Place it next to this notebook or set AUG_SCRIPT correctly."
        )

    subprocess.run([
        "python", str(AUG_SCRIPT),
        "--input-dir", str(RAW_DATA_DIR),
        "--output-dir", str(AUG_DATA_DIR),
        "--seed", str(SEED)
    ], check=True)

DATA_DIR = AUG_DATA_DIR if USE_AUGMENTED_DATA else RAW_DATA_DIR

# ------------------------------
# Load CSV files
# ------------------------------

df_r = pd.read_csv(DATA_DIR / "Real.csv")
df_fh = pd.read_csv(DATA_DIR / "Fake_High.csv")
df_fl = pd.read_csv(DATA_DIR / "Fake_Low.csv")

print("Real:", df_r.shape)
print("Fake High:", df_fh.shape)
print("Fake Low:", df_fl.shape)

# Keep file names for traceability, but train only on numeric columns.
files_r = df_r["file"].copy() if "file" in df_r.columns else pd.Series([f"real_{i}" for i in range(len(df_r))])
files_fh = df_fh["file"].copy() if "file" in df_fh.columns else pd.Series([f"fake_high_{i}" for i in range(len(df_fh))])
files_fl = df_fl["file"].copy() if "file" in df_fl.columns else pd.Series([f"fake_low_{i}" for i in range(len(df_fl))])

X_real = df_r.drop(columns=["file"], errors="ignore").to_numpy(dtype=np.float32)
X_fh = df_fh.drop(columns=["file"], errors="ignore").to_numpy(dtype=np.float32)
X_fl = df_fl.drop(columns=["file"], errors="ignore").to_numpy(dtype=np.float32)

assert X_real.shape[1] == 512, f"Expected 512 columns, got {X_real.shape[1]}"
assert X_fh.shape[1] == 512, f"Expected 512 columns, got {X_fh.shape[1]}"
assert X_fl.shape[1] == 512, f"Expected 512 columns, got {X_fl.shape[1]}"

# Labels: 0=Real, 1=Fake High, 2=Fake Low
X = np.vstack([X_real, X_fh, X_fl]).astype(np.float32)
y = np.concatenate([
    np.zeros(len(X_real), dtype=np.int64),
    np.ones(len(X_fh), dtype=np.int64),
    np.full(len(X_fl), 2, dtype=np.int64)
])
files = pd.concat([files_r, files_fh, files_fl], ignore_index=True)

class_names = ["Real", "Fake High", "Fake Low"]

# Fully shuffle before the 90/10 split, while preserving X/y/file alignment.
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(X))
X = X[perm]
y = y[perm]
files = files.iloc[perm].reset_index(drop=True)

print("Combined X:", X.shape)
print("Combined class counts:", pd.Series(y).value_counts().sort_index().to_dict())


Kefra_Processed_Data/Real.csv -> Kefra_Augmented_Data/Real.csv: 110 rows -> 660 rows
Kefra_Processed_Data/Fake_High.csv -> Kefra_Augmented_Data/Fake_High.csv: 110 rows -> 660 rows
Kefra_Processed_Data/Fake_Low.csv -> Kefra_Augmented_Data/Fake_Low.csv: 120 rows -> 720 rows
Done. Augmented CSVs are in: Kefra_Augmented_Data
Real: (660, 513)
Fake High: (660, 513)
Fake Low: (720, 513)
Combined X: (2040, 512)
Combined class counts: {0: 660, 1: 660, 2: 720}


In [2]:
# ==============================
# Block 2: 1D CNN model, training, testing functions
# ==============================

import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

BATCH_SIZE = 8          # same as paper
LEARNING_RATE = 1e-3    # same as paper
NUM_EPOCHS = 100        # same as paper


def make_activation(name: str):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "leakyrelu":
        return nn.LeakyReLU(negative_slope=0.01)
    if name == "elu":
        return nn.ELU()
    raise ValueError(f"Unknown activation: {name}")


class CNN1DClassifier(nn.Module):
    """Small 1D CNN classifier for 512-point amplitude vectors."""

    def __init__(self, num_classes=3, activation="relu"):
        super().__init__()
        act1 = make_activation(activation)
        act2 = make_activation(activation)
        act3 = make_activation(activation)
        act4 = make_activation(activation)

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),   # 512 -> 256
            act1,
            nn.BatchNorm1d(16),

            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),  # 256 -> 128
            act2,
            nn.BatchNorm1d(32),

            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),  # 128 -> 64
            act3,
            nn.BatchNorm1d(64),

            nn.Conv1d(64, 128, kernel_size=3, stride=2, padding=1), # 64 -> 32
            act4,
            nn.BatchNorm1d(128),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            make_activation(activation),
            nn.Dropout(0.20),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def make_optimizer(name: str, model: nn.Module, lr: float):
    name = name.lower()
    if name == "sgd":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    if name == "adam":
        return optim.Adam(model.parameters(), lr=lr)
    if name == "rmsprop":
        return optim.RMSprop(model.parameters(), lr=lr, momentum=0.9)
    raise ValueError(f"Unknown optimizer: {name}")


def scale_and_tensorize(X_train, X_test):
    # Fit scaler on training only to avoid leakage.
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_test_s = scaler.transform(X_test).astype(np.float32)

    # Conv1d expects [batch, channels, length].
    X_train_t = torch.tensor(X_train_s).unsqueeze(1)
    X_test_t = torch.tensor(X_test_s).unsqueeze(1)
    return X_train_t, X_test_t, scaler


def make_loader(X_tensor, y_array, batch_size=BATCH_SIZE, shuffle=False):
    y_tensor = torch.tensor(y_array, dtype=torch.long)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)


def train_model(X_train, y_train, X_test, y_test, optimizer_name="sgd", activation="relu", verbose=False):
    X_train_t, X_test_t, scaler = scale_and_tensorize(X_train, X_test)
    train_loader = make_loader(X_train_t, y_train, shuffle=True)
    test_loader = make_loader(X_test_t, y_test, shuffle=False)

    model = CNN1DClassifier(num_classes=3, activation=activation).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(optimizer_name, model, LEARNING_RATE)

    train_losses = []

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"Epoch {epoch+1:03d}/{NUM_EPOCHS} | loss={epoch_loss:.6f}")

    y_pred, y_prob = predict_model(model, test_loader)

    return {
        "model": model,
        "scaler": scaler,
        "train_losses": train_losses,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


def predict_model(model, loader):
    model.eval()
    preds = []
    probs = []

    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            logits = model(xb)
            p = torch.softmax(logits, dim=1)
            pred = torch.argmax(p, dim=1)
            preds.extend(pred.cpu().numpy())
            probs.extend(p.cpu().numpy())

    return np.array(preds), np.array(probs)


def summarize_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }


Using device: cpu


In [ ]:
# ==============================
# Block 3: 90/10 test split experiments, matching the paper structure
# ==============================

# Paper-like split: 90% train, 10% test, stratified by class.
X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    X, y, files,
    test_size=0.10,
    random_state=SEED,
    shuffle=True,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train counts:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("Test counts:", pd.Series(y_test).value_counts().sort_index().to_dict())

optimizers_to_test = ["sgd", "adam", "rmsprop"]
activations_to_test = ["relu", "leakyrelu", "elu"]

results_90_10 = []
trained_runs = {}

for opt_name in optimizers_to_test:
    for act_name in activations_to_test:
        print(f"\nTraining 90/10 model: optimizer={opt_name}, activation={act_name}")
        run = train_model(
            X_train, y_train, X_test, y_test,
            optimizer_name=opt_name,
            activation=act_name,
            verbose=False
        )
        y_pred = run["y_pred"]
        metrics = summarize_metrics(y_test, y_pred)
        metrics.update({"optimizer": opt_name, "activation": act_name})
        results_90_10.append(metrics)
        trained_runs[(opt_name, act_name)] = run
        print(metrics)

results_90_10_df = pd.DataFrame(results_90_10).sort_values(
    by=["accuracy", "macro_f1"], ascending=False
).reset_index(drop=True)

results_90_10_df


Train shape: (1836, 512) Test shape: (204, 512)
Train counts: {0: 594, 1: 594, 2: 648}
Test counts: {0: 66, 1: 66, 2: 72}

Training 90/10 model: optimizer=sgd, activation=relu
{'accuracy': 0.8676470588235294, 'macro_precision': 0.8674107142857143, 'macro_recall': 0.8707912457912458, 'macro_f1': 0.8676470588235294, 'optimizer': 'sgd', 'activation': 'relu'}

Training 90/10 model: optimizer=sgd, activation=leakyrelu
{'accuracy': 0.7843137254901961, 'macro_precision': 0.7866161616161617, 'macro_recall': 0.7866161616161617, 'macro_f1': 0.7866161616161617, 'optimizer': 'sgd', 'activation': 'leakyrelu'}

Training 90/10 model: optimizer=sgd, activation=elu
{'accuracy': 0.7794117647058824, 'macro_precision': 0.7847365999539914, 'macro_recall': 0.781986531986532, 'macro_f1': 0.7830749354005168, 'optimizer': 'sgd', 'activation': 'elu'}

Training 90/10 model: optimizer=adam, activation=relu
{'accuracy': 0.8529411764705882, 'macro_precision': 0.8523452643008317, 'macro_recall': 0.8552188552188552, 

In [ ]:
# Detailed report for the best 90/10 model

best = results_90_10_df.iloc[0]
best_key = (best["optimizer"], best["activation"])
best_run = trained_runs[best_key]

print("Best 90/10 configuration:")
print(best)

print("\nClassification report:")
print(classification_report(y_test, best_run["y_pred"], target_names=class_names, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, best_run["y_pred"]))


In [ ]:
# ==============================
# Block 4: 5-fold cross-validation
# ==============================

# This mirrors the paper's 5-fold CV idea, but uses the 1D CNN and CSV data.
# Each fold refits the scaler only on the training fold to avoid leakage.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = []

for opt_name in optimizers_to_test:
    for act_name in activations_to_test:
        print(f"\n5-fold CV: optimizer={opt_name}, activation={act_name}")

        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            run = train_model(
                X_tr, y_tr, X_te, y_te,
                optimizer_name=opt_name,
                activation=act_name,
                verbose=False
            )

            metrics = summarize_metrics(y_te, run["y_pred"])
            metrics.update({
                "optimizer": opt_name,
                "activation": act_name,
                "fold": fold,
            })
            cv_results.append(metrics)
            print(f"  fold={fold} acc={metrics['accuracy']:.4f} f1={metrics['macro_f1']:.4f}")

cv_results_df = pd.DataFrame(cv_results)

cv_summary_df = (
    cv_results_df
    .groupby(["optimizer", "activation"])
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        macro_precision_mean=("macro_precision", "mean"),
        macro_recall_mean=("macro_recall", "mean"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
    )
    .reset_index()
    .sort_values(by=["accuracy_mean", "macro_f1_mean"], ascending=False)
)

cv_summary_df


In [ ]:
# Save results for reporting

results_90_10_df.to_csv("cnn_replica_90_10_results.csv", index=False)
cv_results_df.to_csv("cnn_replica_5fold_raw_results.csv", index=False)
cv_summary_df.to_csv("cnn_replica_5fold_summary.csv", index=False)

print("Saved:")
print("- cnn_replica_90_10_results.csv")
print("- cnn_replica_5fold_raw_results.csv")
print("- cnn_replica_5fold_summary.csv")
